In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Combined12.csv to Combined12.csv


In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# -----------------------------------
# 1. LOAD DATASET
# -----------------------------------

df = pd.read_csv("Combined12.csv")

print("Dataset Shape:", df.shape)

# -----------------------------------
# 2. CHECK DATA
# -----------------------------------

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nTarget Distribution:")
print(df["normalized_label"].value_counts())

# -----------------------------------
# 3. REMOVE DUPLICATES
# -----------------------------------

df = df.drop_duplicates()

# -----------------------------------
# 4. HANDLE MISSING VALUES
# -----------------------------------

df = df.dropna()

# -----------------------------------
# 5. SEPARATE FEATURES AND TARGET
# -----------------------------------

X = df.drop(columns=["normalized_label"])
y = df["normalized_label"]

# -----------------------------------
# 6. TRAIN-TEST SPLIT
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

# -----------------------------------
# 7. FEATURE SCALING
# -----------------------------------

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# -----------------------------------
# 8. CREATE LOGISTIC REGRESSION
# -----------------------------------

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

# -----------------------------------
# 9. TRAIN MODEL
# -----------------------------------

model.fit(X_train_scaled, y_train)

# -----------------------------------
# 10. PREDICTION
# -----------------------------------

y_pred = model.predict(X_test_scaled)

# -----------------------------------
# 11. EVALUATION
# -----------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\nLogistic Regression Accuracy:")
print(accuracy)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Dataset Shape: (533312, 8)

Missing Values:
Pressure            0
global_radiation    0
temp_mean(c)        0
temp_min(c)         0
temp_max(c)         0
Wind_Speed          0
Wind_Bearing        0
normalized_label    0
dtype: int64

Duplicate Rows:
16264

Target Distribution:
normalized_label
0    239241
1    155709
2     87533
3     50829
Name: count, dtype: int64

Training Shape: (413638, 7)
Testing Shape: (103410, 7)

Logistic Regression Accuracy:
0.7935499468136544

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.87      0.84     46201
           1       0.77      0.68      0.72     29926
           2       0.81      0.88      0.84     17260
           3       0.76      0.64      0.70     10023

    accuracy                           0.79    103410
   macro avg       0.79      0.77      0.77    103410
weighted avg       0.79      0.79      0.79    103410


Confusion Matrix:
[[40157  6044     0     0]
 [ 9693 20233     0 

In [ ]:
import joblib

# Save the trained Logistic Regression model
joblib.dump(model, "weather_model.pkl")

# Save the StandardScaler
joblib.dump(scaler, "weather_scaler.pkl")

print("Model and scaler saved successfully!")

Model and scaler saved successfully!


In [ ]:
import joblib
import pandas as pd

# Load the saved model and scaler
loaded_model = joblib.load("weather_model.pkl")
loaded_scaler = joblib.load("weather_scaler.pkl")

# Create one new weather input
new_data = pd.DataFrame([{
    "Pressure": 980.0,
    "global_radiation": 150.0,
    "temp_mean(c)": 20.0,
    "temp_min(c)": 15.0,
    "temp_max(c)": 25.0,
    "Wind_Speed": 5.0,
    "Wind_Bearing": 180
}])

# Apply the same scaling used during training
new_data_scaled = loaded_scaler.transform(new_data)

# Predict
prediction = loaded_model.predict(new_data_scaled)

# Get prediction probabilities
probabilities = loaded_model.predict_proba(new_data_scaled)

print("Predicted Class:", prediction[0])
print("Prediction Probabilities:", probabilities[0])

Predicted Class: 1
Prediction Probabilities: [2.06920998e-090 1.00000000e+000 7.41641482e-122 1.18587818e-033]


In [ ]:
!pip install -q gradio

In [ ]:
import gradio as gr
import numpy as np

# Prediction function
def predict_weather(
    pressure,
    global_radiation,
    temp_mean,
    temp_min,
    temp_max,
    wind_speed,
    wind_bearing
):
    # Arrange the user input in the same order as the training data
    new_data = np.array([[
        pressure,
        global_radiation,
        temp_mean,
        temp_min,
        temp_max,
        wind_speed,
        wind_bearing
    ]])

    # Apply the same scaling used during training
    new_data_scaled = loaded_scaler.transform(new_data)

    # Predict class
    prediction = loaded_model.predict(new_data_scaled)[0]

    # Get probabilities
    probabilities = loaded_model.predict_proba(new_data_scaled)[0]

    # Create probability display
    probability_text = "\n".join([
        f"Class {i}: {probabilities[i] * 100:.2f}%"
        for i in range(len(probabilities))
    ])

    return f"Predicted Class: {prediction}", probability_text


# Create the UI
with gr.Blocks() as demo:

    gr.Markdown("# 🌦️ Weather Classification using Logistic Regression")
    gr.Markdown(
        "Enter the weather parameters below and click **Predict**."
    )

    with gr.Row():
        pressure = gr.Number(
            label="Pressure",
            value=980.0
        )

        global_radiation = gr.Number(
            label="Global Radiation",
            value=150.0
        )

    with gr.Row():
        temp_mean = gr.Number(
            label="Mean Temperature (°C)",
            value=20.0
        )

        temp_min = gr.Number(
            label="Minimum Temperature (°C)",
            value=15.0
        )

        temp_max = gr.Number(
            label="Maximum Temperature (°C)",
            value=25.0
        )

    with gr.Row():
        wind_speed = gr.Number(
            label="Wind Speed",
            value=5.0
        )

        wind_bearing = gr.Number(
            label="Wind Bearing",
            value=180
        )

    predict_button = gr.Button("🔮 Predict")

    prediction_output = gr.Textbox(
        label="Prediction"
    )

    probability_output = gr.Textbox(
        label="Class Probabilities"
    )

    predict_button.click(
        fn=predict_weather,
        inputs=[
            pressure,
            global_radiation,
            temp_mean,
            temp_min,
            temp_max,
            wind_speed,
            wind_bearing
        ],
        outputs=[
            prediction_output,
            probability_output
        ]
    )


# Launch the UI
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b404a4ac70c6a5c742.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
print(df.columns.tolist())

['Pressure', 'global_radiation', 'temp_mean(c)', 'temp_min(c)', 'temp_max(c)', 'Wind_Speed', 'Wind_Bearing', 'normalized_label']
